Import library

In [110]:
import mne
import os
import pandas as pd
import numpy as np
from scipy.signal import welch

root = r'D:\MyProjects\EEGDatasets\InternalAttention'
subject = 'sub-001'
eeg_dir = os.path.join(root, subject, 'eeg')
set_file = os.path.join(eeg_dir, f'{subject}_task-internalattention_eeg.set')
save_file = f'{subject}_task-internalattention_128hz_raw.fif'

Down sampling

In [111]:
raw = mne.io.read_raw_eeglab(set_file, preload=True)

print('Original sfreq:', raw.info['sfreq'])
print('Original shape:', raw.get_data().shape)

raw.filter(l_freq=1.0, h_freq=45.0)
raw.notch_filter(freqs=60)
raw.resample(128)

print('Resampled sfreq:', raw.info['sfreq'])
print('Resampled shape:', raw.get_data().shape)

raw.save(save_file, overwrite=True)

print('Done.')

Original sfreq: 250.0
Original shape: (32, 182331)
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 825 samples (3.300 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 d

Compute band power

In [112]:
# ========= 1. read data =========
raw = mne.io.read_raw_fif(
    fr'{subject}_task-internalattention_128hz_raw.fif',
    preload=True
)

print('Current sfreq:', raw.info['sfreq'])
print('Current shape:', raw.get_data().shape)
print('Current channels:', raw.ch_names)

# ========= 2. mapping channels =========
epoc_mapping = {
    'AF3': 'E1',
    'AF4': 'E2',
    'F3':  'E3',
    'F4':  'E4',
    'FC5': 'E11',
    'FC6': 'E12',
    'F7':  'E31',
    'F8':  'E32',
}

selected_channels = list(epoc_mapping.values())
raw.pick(selected_channels)

print('Selected channels:', raw.ch_names)
print('Shape after channel selection:', raw.get_data().shape)

# ========= 3. read events.tsv =========
events_file = fr'D:\MyProjects\EEGDatasets\InternalAttention\{subject}\eeg\{subject}_task-internalattention_events.tsv'
events_df = pd.read_csv(events_file, sep='\t')

print('\nOriginal event columns:')
print(events_df.columns.tolist())

print('\nOriginal trial_type counts:')
print(events_df['trial_type'].value_counts(dropna=False))

# ========= 4. ignore wait; keep relax; map getready and concentrate to concentrate =========
events_df = events_df[events_df['trial_type'] != 'wait'].copy()

events_df['trial_type'] = events_df['trial_type'].replace({
    'getready': 'getready',
    'concentrate': 'concentrate',
    'relax': 'relax'
})

print('\nFiltered / remapped trial_type counts:')
print(events_df['trial_type'].value_counts())

# ========= 5. keep field =========
keep_cols = [c for c in ['onset', 'duration', 'trial_type', 'value'] if c in events_df.columns]
events_df = events_df[keep_cols].copy()

# ========= 6. Convert to sampling points =========
sfreq = raw.info['sfreq']
events_df['sample'] = (events_df['onset'] * sfreq).round().astype(int)

events_df = events_df.sort_values('sample').reset_index(drop=True)

print('\nFiltered events preview:')
print(events_df.head(20))

# ========= 7. Save the filtered events =========
save_events_path = f'{subject}_relax_concentrate_events_128hz.csv'
events_df.to_csv(save_events_path, index=False)
print('\nSaved filtered events to', save_events_path)

# ========= 8. Merge consecutive intervals by trial_type =========

segments = []

n_times = raw.n_times

current_label = events_df.loc[0, 'trial_type']
current_start = int(events_df.loc[0, 'sample'])

for i in range(1, len(events_df)):
    label = events_df.loc[i, 'trial_type']
    sample = int(events_df.loc[i, 'sample'])


    if label != current_label:
        segments.append({
            'label': current_label,
            'start_sample': current_start,
            'end_sample': sample
        })
        current_label = label
        current_start = sample


segments.append({
    'label': current_label,
    'start_sample': current_start,
    'end_sample': n_times
})

segments_df = pd.DataFrame(segments)
segments_df['start_time_sec'] = segments_df['start_sample'] / sfreq
segments_df['end_time_sec'] = segments_df['end_sample'] / sfreq
segments_df['duration_sec'] = segments_df['end_time_sec'] - segments_df['start_time_sec']

print('\nMerged segments:')
print(segments_df)

segments_save_path = f'{subject}_segments_128hz.csv'
segments_df.to_csv(segments_save_path, index=False)
print('\nSaved merged segments to', segments_save_path)

# ========= 9. define band power =========
bands = {
    'theta': (4, 8),
    'alpha': (8, 12),
    'betaL': (12, 16),
    'betaH': (16, 25),
    'gamma': (25, 45),
}

# ========= 10. compute band power =========
def compute_band_powers(segment, sfreq, bands, ch_names):
    """
    segment: shape [n_channels, n_samples]
    return: dict
    """
    features = {}

    freqs, psd = welch(
        segment,
        fs=sfreq,
        nperseg=segment.shape[1],
        axis=1
    )

    for band_name, (fmin, fmax) in bands.items():
        idx = (freqs >= fmin) & (freqs < fmax)

        if np.sum(idx) == 0:
            band_power = np.zeros(segment.shape[0])
        else:
            band_power = np.trapezoid(psd[:, idx], freqs[idx], axis=1)

        for ch_i, ch in enumerate(ch_names):
            features[f'{ch}_{band_name}'] = band_power[ch_i]

    return features

# ========= 11. Divide each major interval into 2-second windows =========
data = raw.get_data()
ch_names = raw.ch_names

window_sec = 2.0
window_size = int(window_sec * sfreq)   # 128 Hz -> 256 samples
step_size = window_size                 # no overlap; 50% overlap -> window_size // 2

rows = []

for _, seg in segments_df.iterrows():
    label = seg['label']

    # skip getready
    if label == 'getready':
        continue

    seg_start = int(seg['start_sample'])
    seg_end = int(seg['end_sample'])

    seg_len = seg_end - seg_start
    if seg_len < window_size:
        continue

    for win_start in range(seg_start, seg_end - window_size + 1, step_size):
        win_end = win_start + window_size

        if win_end > seg_end:
            continue

        segment = data[:, win_start:win_end]

        feat = compute_band_powers(segment, sfreq, bands, ch_names)

        feat['label'] = label
        feat['start_sample'] = win_start
        feat['end_sample'] = win_end
        feat['start_time_sec'] = win_start / sfreq
        feat['end_time_sec'] = win_end / sfreq
        feat['window_sec'] = window_sec

        rows.append(feat)

features_df = pd.DataFrame(rows)

print('\nBand power features shape:', features_df.shape)
print(features_df.head())

print('\nWindow label counts:')
print(features_df['label'].value_counts())

# ========= 12. Save the band power features =========
features_save_path = f'{subject}_bandpower_2s_relax_concentrate.csv'
features_df.to_csv(features_save_path, index=False)
print('\nSaved band power features to', features_save_path)

print('\nWindow label counts:')
print(features_df['label'].value_counts())

print('\nDone.')

Opening raw data file sub-001_task-internalattention_128hz_raw.fif...
    Range : 0 ... 93352 =      0.000 ...   729.312 secs
Ready.
Reading 0 ... 93352  =      0.000 ...   729.312 secs...
Current sfreq: 128.0
Current shape: (32, 93353)
Current channels: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29', 'E30', 'E31', 'E32']
Selected channels: ['E1', 'E2', 'E3', 'E4', 'E11', 'E12', 'E31', 'E32']
Shape after channel selection: (8, 93353)

Original event columns:
['onset', 'duration', 'sample', 'trial_type', 'response_time', 'stim_file', 'value', 'HED', 'second_after_ready']

Original trial_type counts:
trial_type
concentrate    300
relax          211
getready       120
wait            16
Name: count, dtype: int64

Filtered / remapped trial_type counts:
trial_type
concentrate    300
relax          211
getready       120
Name: count, dtype: int64


Compute band ratio

In [118]:
raw = mne.io.read_raw_fif(
    fr'{subject}_task-internalattention_128hz_raw.fif',
    preload=False
)


target_channels = ['E1','E2','E3','E4','E11','E12','E31','E32']
if set(raw.ch_names) != set(target_channels):
    raw.pick(target_channels)

channels = raw.ch_names
print('Channels used:', channels)


features_df = pd.read_csv(f'{subject}_bandpower_2s_relax_concentrate.csv')

print('features_df shape:', features_df.shape)
print(features_df.head())


ratio_rows = []

for _, row in features_df.iterrows():
    feat = {}

    for ch in channels:
        theta = row[f'{ch}_theta']
        alpha = row[f'{ch}_alpha']
        betaL = row[f'{ch}_betaL']
        betaH = row[f'{ch}_betaH']
        gamma = row[f'{ch}_gamma']

        beta = betaL + betaH

        eps = 1e-8

        # feat[f'{ch}_beta_alpha'] = beta / (alpha + eps)
        # feat[f'{ch}_beta_alpha_theta'] = beta / (alpha + theta + eps)
        # feat[f'{ch}_theta_beta'] = theta / (beta + eps)
        # feat[f'{ch}_theta_alpha_beta'] = theta / (alpha + beta + eps)
        # feat[f'{ch}_log_inv_alpha'] = np.log(1.0 / (alpha + eps))

        feat[f'{ch}_theta'] = theta
        feat[f'{ch}_alpha'] = alpha
        # feat[f'{ch}_beta'] = beta
        feat[f'{ch}_betaL'] = betaL
        feat[f'{ch}_betaH'] = betaH
        feat[f'{ch}_gamma'] = gamma


    feat['label'] = row['label']

    ratio_rows.append(feat)

ratio_df = pd.DataFrame(ratio_rows)

print('\nratio_df shape:', ratio_df.shape)
print(ratio_df.head())


ratio_df.to_csv(f'{subject}_features.csv', index=False)

print(f'\nSaved to {subject}_features.csv')

Opening raw data file sub-001_task-internalattention_128hz_raw.fif...
    Range : 0 ... 93352 =      0.000 ...   729.312 secs
Ready.
Channels used: ['E1', 'E2', 'E3', 'E4', 'E11', 'E12', 'E31', 'E32']
features_df shape: (253, 46)
       E1_theta      E2_theta      E3_theta      E4_theta     E11_theta  \
0  2.740419e-12  4.262935e-12  1.667305e-12  4.142364e-12  3.701100e-12   
1  3.859187e-12  8.076384e-12  2.175997e-12  5.315521e-12  2.747891e-12   
2  1.043377e-11  9.478539e-12  4.330296e-12  5.232470e-12  1.920597e-11   
3  8.351585e-12  2.288468e-12  5.339798e-12  1.854389e-12  2.129326e-11   
4  4.230157e-12  4.358245e-12  1.368214e-12  3.059857e-12  4.550737e-12   

      E12_theta     E31_theta     E32_theta      E1_alpha      E2_alpha  ...  \
0  2.742021e-12  6.883965e-12  4.324185e-12  9.609919e-12  1.489430e-11  ...   
1  7.487675e-12  6.562457e-12  1.136660e-11  7.007997e-12  6.112480e-12  ...   
2  8.569767e-12  2.204183e-11  1.369276e-11  5.367570e-12  9.843288e-12  ...   

Train

In [119]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ========= 1. read data =========
df = pd.read_csv(f'{subject}_features.csv')

print('Data shape:', df.shape)
print(df.head())

# ========= 2. Separate features and labels =========
X = df.drop(columns=['label'])
y = df['label']
feature_names = X.columns.tolist()
print('\nLabel distribution:')
print(y.value_counts())

# ========= 3. split train / test =========
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ========= 4. standardization =========
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ========= 5. Logistic Regression =========
model = LogisticRegression(
    penalty='l2',
    C=1.0,
    max_iter=1000,
    solver='lbfgs'
)

model.fit(X_train, y_train)

# ========= 6. predict =========
y_pred = model.predict(X_test)

# ========= 7. evaluate =========
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print('\nAccuracy:', acc)
print('\nClassification Report:')
print(report)
print('\nConfusion Matrix:')
print(cm)

# ========= 8. write results =========
results_path = 'results_no_ratio.txt'

with open(results_path, 'w', encoding='utf-8') as f:
    f.write(f'Subject: {subject}\n')
    f.write(f'Data shape: {df.shape}\n\n')

    f.write('Label distribution:\n')
    f.write(y.value_counts().to_string())
    f.write('\n\n')

    f.write(f'Accuracy: {acc}\n\n')

    f.write('Classification Report:\n')
    f.write(report)
    f.write('\n')

    f.write('Confusion Matrix:\n')
    f.write(str(cm))
    f.write('\n')

print(f'\nSaved results to {results_path}')

# ========= 9. save model =========
save_model = f'{subject}_logreg_model.joblib'
save_scaler = f'{subject}_scaler.joblib'
save_meta = f'{subject}_feature_names.joblib'

joblib.dump(model, save_model)
joblib.dump(scaler, save_scaler)
joblib.dump(feature_names, save_meta)

print(f'Saved model to {save_model}')
print(f'Saved scaler to {save_scaler}')
print(f'Saved feature names to {save_meta}')

Data shape: (253, 41)
       E1_theta      E1_alpha      E1_betaL      E1_betaH      E1_gamma  \
0  2.740419e-12  9.609919e-12  2.321935e-12  3.185292e-12  4.503614e-12   
1  3.859187e-12  7.007997e-12  2.068480e-12  4.062212e-12  2.365995e-12   
2  1.043377e-11  5.367570e-12  1.505781e-12  1.989660e-12  2.648806e-12   
3  8.351585e-12  1.084012e-11  2.309547e-12  2.618220e-12  2.832202e-12   
4  4.230157e-12  6.228017e-12  1.228341e-12  3.150616e-12  4.552605e-12   

       E2_theta      E2_alpha      E2_betaL      E2_betaH      E2_gamma  ...  \
0  4.262935e-12  1.489430e-11  2.524582e-12  3.801744e-12  5.805303e-12  ...   
1  8.076384e-12  6.112480e-12  4.047565e-12  4.931460e-12  4.112103e-12  ...   
2  9.478539e-12  9.843288e-12  1.161879e-12  1.432675e-12  1.782265e-12  ...   
3  2.288468e-12  1.083378e-11  1.824813e-12  2.191147e-12  3.114609e-12  ...   
4  4.358245e-12  1.165689e-11  9.164725e-13  3.096025e-12  3.735078e-12  ...   

      E31_alpha     E31_betaL     E31_betaH   

Load model and predict

In [126]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = joblib.load(f'{subject}_logreg_model.joblib')
scaler = joblib.load(f'{subject}_scaler.joblib')
feature_names = joblib.load(f'{subject}_feature_names.joblib')

df_new = pd.read_csv('sub-005/sub-005_features.csv')

X_new = df_new[feature_names]
y_true = df_new['label']

X_new_scaled = scaler.transform(X_new)
y_pred = model.predict(X_new_scaled)

print('Predictions:')
print(y_pred[:10])

acc = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print('\nAccuracy:', acc)

print('\nClassification Report:')
print(report)

print('\nConfusion Matrix:')
print(cm)

Predictions:
['concentrate' 'concentrate' 'concentrate' 'concentrate' 'concentrate'
 'concentrate' 'concentrate' 'concentrate' 'concentrate' 'concentrate']

Accuracy: 0.5594059405940595

Classification Report:
              precision    recall  f1-score   support

 concentrate       0.54      0.96      0.69       210
       relax       0.75      0.12      0.21       194

    accuracy                           0.56       404
   macro avg       0.65      0.54      0.45       404
weighted avg       0.64      0.56      0.46       404


Confusion Matrix:
[[202   8]
 [170  24]]
